# 07 — TMDB Snapshot Gap Analysis

Cross-references the Kaggle snapshot against the current TMDB daily export to identify:

1. **Deleted movies** — in Kaggle but removed from TMDB (likely spam/duplicates)
2. **New movies** — added to TMDB after the Kaggle snapshot
3. **Keyword noise** — keywords sourced only from deleted movies
4. **Keyword gap** — valid movies with no keywords (candidates for API enrichment)

| | Count |
|---|---|
| Kaggle snapshot | 1,382,594 |
| TMDB current export | 1,168,807 |
| In both | 1,167,378 |
| Deleted from TMDB | 214,017 |
| New since snapshot | 1,429 |
| Valid movies with no keywords | 884,707 |

In [1]:
from collections import Counter
from pathlib import Path

import pandas as pd

KAGGLE_FILE = Path('/Users/bobby/.cache/kagglehub/datasets/asaniczka/tmdb-movies-dataset-2023-930k-movies/versions/868/TMDB_movie_dataset_v11.csv')
TMDB_EXPORT = Path('data/tmdb_movie_ids.csv')
DATA = Path('data')

kaggle = pd.read_csv(KAGGLE_FILE, low_memory=False)
tmdb   = pd.read_csv(TMDB_EXPORT)

print(f'Kaggle: {len(kaggle):,}  TMDB export: {len(tmdb):,}')

Kaggle: 1,382,594  TMDB export: 1,168,807


In [2]:
kaggle_ids = set(kaggle['id'].dropna().astype(int))
tmdb_ids   = set(tmdb['id'].dropna().astype(int))

valid_ids   = kaggle_ids & tmdb_ids    # still live in TMDB
deleted_ids = kaggle_ids - tmdb_ids    # removed from TMDB
new_ids     = tmdb_ids - kaggle_ids    # added after snapshot

print(f'Valid (in both):        {len(valid_ids):,}')
print(f'Deleted from TMDB:      {len(deleted_ids):,}')
print(f'New since snapshot:     {len(new_ids):,}')

Valid (in both):        1,167,378
Deleted from TMDB:      214,017
New since snapshot:     1,429


## Keyword noise — from deleted movies only

In [3]:
def keyword_counter(df):
    c = Counter()
    for cell in df['keywords'].dropna():
        for kw in str(cell).split(', '):
            kw = kw.strip()
            if kw:
                c[kw] += 1
    return c

deleted_kws = keyword_counter(kaggle[kaggle['id'].isin(deleted_ids)])
valid_kws   = keyword_counter(kaggle[kaggle['id'].isin(valid_ids)])

deleted_only = set(deleted_kws) - set(valid_kws)

print(f'Keywords from valid movies:          {len(valid_kws):,}')
print(f'Keywords only from deleted movies:   {len(deleted_only):,}  ← noise candidates')
print(f'Total unique keywords in Kaggle:     {len(set(deleted_kws)|set(valid_kws)):,}')

Keywords from valid movies:          63,607
Keywords only from deleted movies:   4,309  ← noise candidates
Total unique keywords in Kaggle:     67,916


In [4]:
# Save noise keyword list
noise_df = pd.DataFrame([
    {'keyword': kw, 'deleted_movie_count': deleted_kws[kw]}
    for kw in deleted_only
]).sort_values('deleted_movie_count', ascending=False).reset_index(drop=True)

noise_df.to_csv(DATA / 'tmdb_keywords_deleted_only.csv', index=False)
print(f'Saved {len(noise_df):,} noise keywords')
noise_df.head(20)

Saved 4,309 noise keywords


,keyword,deleted_movie_count
0,creampie,1529
1,big butt,916
2,double penetration,705
3,natural tits,629
4,muscled men,427
5,cum swallowing,404
6,petite,392
7,tit fucking,354
8,deep throat,338
9,cumshots,316


## Keyword gap — valid movies with no keywords

In [5]:
valid_df = kaggle[kaggle['id'].isin(valid_ids)].copy()
valid_df['has_keywords'] = valid_df['keywords'].notna() & (valid_df['keywords'].str.strip() != '')

no_kw = valid_df[~valid_df['has_keywords']]
print(f'Valid movies with keywords:    {valid_df["has_keywords"].sum():,} ({valid_df["has_keywords"].mean()*100:.1f}%)')
print(f'Valid movies WITHOUT keywords: {len(no_kw):,}')

# Save the no-keyword movie IDs + titles for potential API enrichment
no_kw_out = no_kw[['id', 'title', 'release_date']].copy()
no_kw_out = no_kw_out.merge(tmdb[['id', 'popularity']], on='id', how='left')
no_kw_out = no_kw_out.sort_values('popularity', ascending=False).reset_index(drop=True)
no_kw_out.to_csv(DATA / 'tmdb_movies_no_keywords.csv', index=False)
print(f'Saved → data/tmdb_movies_no_keywords.csv')
no_kw_out.head(10)

Valid movies with keywords:    283,628 (24.3%)
Valid movies WITHOUT keywords: 884,707


Saved → data/tmdb_movies_no_keywords.csv


,id,title,release_date,popularity
0,1265609,War Machine,NaN,698.6814
1,1290821,Untitled Baltasar Kormákur/Jason Statham movie,NaN,344.3062
2,799882,The Bluff,NaN,243.2530
3,1327819,Hoppers,2026-03-06,144.3679
4,840464,Greenland: Migration,NaN,138.2360
5,1088434,Hellfire,NaN,129.2729
6,1634301,Vanaveera,2026-02-13,124.9390
7,1084242,Zootopia 2,NaN,120.9657
8,1168190,The Wrecking Crew,NaN,114.6537
9,1419406,The Shadow's Edge,NaN,111.2333


## Valid movie IDs — clean list for API work

In [6]:
# Save clean valid movie ID list (for targeted keyword refresh)
valid_export = tmdb[tmdb['id'].isin(valid_ids)].sort_values('popularity', ascending=False).reset_index(drop=True)
valid_export.to_csv(DATA / 'tmdb_movie_ids_valid.csv', index=False)
print(f'Valid movie IDs saved: {len(valid_export):,} → data/tmdb_movie_ids_valid.csv')

Valid movie IDs saved: 1,167,378 → data/tmdb_movie_ids_valid.csv
